# Metrics and Evaluation - California Housing Dataset

In [139]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import gzip
from pathlib import Path
import seaborn as sns
from sklearn.datasets import fetch_california_housing

In [140]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [141]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [142]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


In [143]:
from sklearn.model_selection import train_test_split

x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

In [144]:
# Todas columnas en dataset
cols = list(x_train.columns)
print(cols)

# Todas menos lat/lon para outliers
preprocess_cols = []
for i, col in enumerate(cols):
  if col != "Latitude" and col != "Longitude":
    preprocess_cols.append(i)

names_outliers_cols = []
# Columnas a trabajar con outliers
for i in range(len(preprocess_cols)):
    names_outliers_cols.append(cols[preprocess_cols[i]])
print(names_outliers_cols)

['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']


In [145]:
class California(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X.values, dtype=torch.float32)
    self.y = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

  def __len__(self):
    return len(self.y)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [146]:
train_data = California(x_train, y_train)
val_data = California(x_val, y_val)
test_data = California(x_test, y_test)

print(f"Train data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Test data size: {len(test_data)}")

Train data size: 16512
Validation data size: 2064
Test data size: 2064


## **Preprocesmiento**

In [147]:
class OutlierLayer(nn.Module):
    def __init__(self, lower_bound, upper_bound, preprocess_cols):
        super().__init__()

        # Indx de Columnas donde aplicamos outliers (No Lat/Long)
        self.preprocess_cols = preprocess_cols

        # Guardar límites como buffers
        self.register_buffer("lower_bound", lower_bound)
        self.register_buffer("upper_bound", upper_bound)

    def forward(self, X):
        X = X.clone()

        # Clipping
        for i, preprocess_cols in enumerate(self.preprocess_cols):
            X[:, preprocess_cols] = torch.clamp(
                X[:, preprocess_cols],
                min=self.lower_bound[i],
                max=self.upper_bound[i]
            )

        return X

In [148]:
class ScalingLayer(nn.Module):
    def __init__(self, mean, std, preprocess_cols):
        super().__init__()

        # Indx de Columnas donde aplicamos outliers (No Lat/Long)
        self.preprocess_cols = preprocess_cols

        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, X):
        X_scaled = X.clone()
        X_scaled[:, self.preprocess_cols] = (
            X[:, self.preprocess_cols] - self.mean) / (self.std + 1e-8)
        return X_scaled

In [149]:
# Calculo de Outliers y Bounds
X_train_tensor = train_data.X

Q1 = torch.quantile(X_train_tensor[:, preprocess_cols], 0.25, dim=0)
Q3 = torch.quantile(X_train_tensor[:, preprocess_cols], 0.75, dim=0)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Scaling
mean = X_train_tensor[:, preprocess_cols].mean(dim=0)
std = X_train_tensor[:, preprocess_cols].std(dim=0)

## **Model Training**

In [150]:
def train(_model, _train_loader, _val_loader, _criterion, _optimizer, _num_epochs, patience, delta):
  training_losses = []
  validation_losses = []

  val_loss_history = []

  for epoch in range(_num_epochs):
    _model.train()
    running_loss = 0.0

    for X_batch, y_batch in tqdm(_train_loader, desc=f"Epoch {epoch + 1}/{_num_epochs}"):
      _optimizer.zero_grad()
      outputs = _model(X_batch)
      loss = _criterion(outputs, y_batch)
      loss.backward()
      _optimizer.step()
      running_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = running_loss / len(_train_loader.dataset)
    training_losses.append(epoch_train_loss)

    _model.eval()
    val_loss = 0.0
    with torch.no_grad():
      for X_val, y_val in _val_loader:
        val_outputs = _model(X_val)
        val_loss += _criterion(val_outputs, y_val).item() * X_val.size(0)

    epoch_val_loss = val_loss / len(_val_loader.dataset)
    validation_losses.append(epoch_val_loss)
    val_loss_history.append(epoch_val_loss)

    print(f"Epoch {epoch+1}: "
          f"Train Loss: {epoch_train_loss:.4f}, "
          f"Val Loss: {epoch_val_loss:.4f}\n")

    # Early stopping con delta
    if len(val_loss_history) > patience:
      loss_diff = abs(val_loss_history[-patience-1] - val_loss_history[-1])

      if loss_diff < delta and val_loss_history[-patience-1] >= val_loss_history[-1]:
        print(f"Early stopping triggered at epoch {epoch+1} (delta={delta}, patience={patience})")
        break

  return training_losses, validation_losses

In [151]:
# MODELO 1
model1 = nn.Sequential(
    OutlierLayer(lower_bound, upper_bound, preprocess_cols),
    ScalingLayer(mean, std, preprocess_cols),

    nn.Linear(x_train.shape[1], 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(device)

criterion = nn.MSELoss()

optimizer1 = optim.Adam(model1.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 10
patience = 5
delta = 0.001
batch_size = 100

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=batch_size)
test_loader  = DataLoader(test_data, batch_size=batch_size)

train_losses1, val_losses1 = train(
    model1,
    train_loader,
    val_loader,
    criterion,
    optimizer1,
    num_epochs,
    patience,
    delta
)

Epoch 1/10: 100%|██████████| 166/166 [00:00<00:00, 352.90it/s]


Epoch 1: Train Loss: 1.3153, Val Loss: 0.9598



Epoch 2/10: 100%|██████████| 166/166 [00:00<00:00, 312.20it/s]


Epoch 2: Train Loss: 0.7259, Val Loss: 0.6627



Epoch 3/10: 100%|██████████| 166/166 [00:00<00:00, 365.00it/s]


...

Epoch 8: Train Loss: 0.4972, Val Loss: 0.5266



Epoch 9/10: 100%|██████████| 166/166 [00:00<00:00, 272.71it/s]


Epoch 9: Train Loss: 0.5240, Val Loss: 0.5472



Epoch 10/10: 100%|██████████| 166/166 [00:00<00:00, 180.15it/s]

Epoch 10: Train Loss: 0.4857, Val Loss: 0.5696



In [152]:
# MODELO 2
model2 = nn.Sequential(
    OutlierLayer(lower_bound, upper_bound, preprocess_cols),
    ScalingLayer(mean, std, preprocess_cols),

    nn.Linear(x_train.shape[1], 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(device)

criterion = nn.MSELoss()

optimizer2 = optim.Adam(model2.parameters(), lr=0.005, weight_decay=1e-4)

num_epochs = 30
patience = 5
delta = 0.001
batch_size = 130

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=batch_size)
test_loader  = DataLoader(test_data, batch_size=batch_size)

train_losses2, val_losses2 = train(
    model2,
    train_loader,
    val_loader,
    criterion,
    optimizer2,
    num_epochs,
    patience,
    delta
)

Epoch 1/30: 100%|██████████| 128/128 [00:00<00:00, 223.76it/s]


Epoch 1: Train Loss: 2.0678, Val Loss: 0.9611



Epoch 2/30: 100%|██████████| 128/128 [00:00<00:00, 220.26it/s]


Epoch 2: Train Loss: 0.6252, Val Loss: 0.5626



Epoch 3/30: 100%|██████████| 128/128 [00:00<00:00, 155.45it/s]


...

Epoch 28: Train Loss: 0.4320, Val Loss: 0.7760



Epoch 29/30: 100%|██████████| 128/128 [00:01<00:00, 76.83it/s]


Epoch 29: Train Loss: 0.4538, Val Loss: 0.4749



Epoch 30/30: 100%|██████████| 128/128 [00:02<00:00, 62.33it/s]


Epoch 30: Train Loss: 0.4440, Val Loss: 0.7106



In [153]:
# MODELO 3
model3 = nn.Sequential(
    OutlierLayer(lower_bound, upper_bound, preprocess_cols),
    ScalingLayer(mean, std, preprocess_cols),

    nn.Linear(x_train.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(device)

criterion = nn.MSELoss()

optimizer3 = optim.Adam(model3.parameters(), lr=0.01, weight_decay=1e-4)

num_epochs = 15
patience = 5
delta = 0.001
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=batch_size)
test_loader  = DataLoader(test_data, batch_size=batch_size)

train_losses3, val_losses3 = train(
    model3,
    train_loader,
    val_loader,
    criterion,
    optimizer3,
    num_epochs,
    patience,
    delta
)

Epoch 1/15: 100%|██████████| 258/258 [00:00<00:00, 363.92it/s]


Epoch 1: Train Loss: 2.2886, Val Loss: 0.7153



Epoch 2/15: 100%|██████████| 258/258 [00:00<00:00, 472.58it/s]


Epoch 2: Train Loss: 0.5449, Val Loss: 0.5051



Epoch 3/15: 100%|██████████| 258/258 [00:00<00:00, 552.16it/s]


...

Epoch 13: Train Loss: 0.4819, Val Loss: 0.4660



Epoch 14/15: 100%|██████████| 258/258 [00:00<00:00, 484.90it/s]


Epoch 14: Train Loss: 0.4606, Val Loss: 0.4860



Epoch 15/15: 100%|██████████| 258/258 [00:00<00:00, 464.34it/s]

Epoch 15: Train Loss: 0.4657, Val Loss: 0.4579



## **Evaluation**

In [154]:
def evaluate_model(model, test_loader, criterion):
    model.eval()
    test_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            test_loss += loss.item() * X_batch.size(0)

    test_loss /= len(test_loader.dataset)
    rmse = np.sqrt(test_loss)

    return test_loss, rmse

test_loss1, rmse1 = evaluate_model(model1, test_loader, criterion)
test_loss2, rmse2 = evaluate_model(model2, test_loader, criterion)
test_loss3, rmse3 = evaluate_model(model3, test_loader, criterion)

print(f"Modelo 1 - Test Loss: {test_loss1:.4f}, RMSE: {rmse1:.4f}")
print(f"Modelo 2 - Test Loss: {test_loss2:.4f}, RMSE: {rmse2:.4f}")
print(f"Modelo 3 - Test Loss: {test_loss3:.4f}, RMSE: {rmse3:.4f}")

best_model = model1
best_rmse = rmse1
best_name = "Modelo 1"

if rmse2 < best_rmse:
    best_model = model2
    best_rmse = rmse2
    best_name = "Modelo 2"

if rmse3 < best_rmse:
    best_model = model3
    best_rmse = rmse3
    best_name = "Modelo 3"

print(f"\nMejor modelo: {best_name} con RMSE = {best_rmse:.4f}")

Modelo 1 - Test Loss: 0.5890, RMSE: 0.7675
Modelo 2 - Test Loss: 0.7241, RMSE: 0.8509
Modelo 3 - Test Loss: 0.4627, RMSE: 0.6802

Mejor modelo: Modelo 3 con RMSE = 0.6802


## **Reflexion**

Gracias a este ejercicio puse en práctica los conocimientos que ya había adquirido con PyTorch y las diferencias que existen con sklearn, pero ahora aplicado a un problema de regresión. Afortunadamente ya se conocía el dataset, pero el cuidado que se debe tener para no realizar preprocesamiento en columnas equivocadas o provocar data leakage es muy importante, sobre todo para entender el contexto de lo que se está haciendo, como en el caso de la longitud y latitud. Sin duda, esto me servirá a futuro para seguir desarrollando mis habilidades al entrenar modelos y trabajar con diferentes herramientas.